# Stretch Pick Cube — Week 4 Visualization
Cross-embodiment dataset: Stretch 3 mobile manipulator with P-controller navigation

In [ ]:
import numpy as np, matplotlib.pyplot as plt, json, random
from pathlib import Path
DATA = Path('/home/shareef/robot_zoo/data/demos/stretch_pick_cube')
meta = json.load(open(DATA/'metadata.json'))
episodes = sorted(DATA.glob('episode_*.npz'))
print('Task:', meta['task'], '| Robot:', meta['robot'])
print('Episodes:', meta['total_episodes'], '| Success rate:', meta['success_rate'])
print('Files on disk:', len(episodes))
ep0 = np.load(episodes[0])
print('Keys:', list(ep0.keys()))
print('observation_state:', ep0['observation_state'].shape)

In [ ]:
# Joint trajectories — key joints for Stretch pick task
joint_names = ['left_wheel','right_wheel','lift','head_pan','head_tilt',
               'arm_l3','arm_l2','arm_l1','arm_l0']
sample = random.sample(episodes, min(3, len(episodes)))
fig, axes = plt.subplots(3, 1, figsize=(14, 10))
for ax, path in zip(axes, sample):
    ep = np.load(path)
    jp = ep['observation_state']
    for j, name in enumerate(joint_names):
        ax.plot(jp[:, j], label=name, alpha=0.8, linewidth=1.5)
    ax.set_title(f'{path.name} | steps: {len(jp)}')
    ax.set_ylabel('Joint pos (rad/m)'); ax.legend(fontsize=7, ncol=5); ax.grid(alpha=0.3)
plt.suptitle('Stretch Pick Cube — Joint Trajectories (9 joints)', fontsize=13)
plt.tight_layout()
plt.savefig(DATA/'joint_trajectories.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Key phase joints: lift and arm extension
ep = np.load(episodes[0])
jp = ep['observation_state']
fig, axes = plt.subplots(2, 1, figsize=(12, 7))
axes[0].plot(jp[:, 2], color='steelblue', linewidth=2.5, label='lift')
axes[0].axhline(0.18, color='orange', linestyle='--', alpha=0.7, label='extend height (0.18m)')
axes[0].axhline(0.0,  color='red',    linestyle='--', alpha=0.7, label='grasp height (0.0m)')
axes[0].axhline(0.4,  color='green',  linestyle='--', alpha=0.7, label='lift height (0.4m)')
axes[0].set_title('Lift Phase Trajectory'); axes[0].set_ylabel('Height (m)')
axes[0].legend(); axes[0].grid(alpha=0.3)
for i, (idx, name) in enumerate(zip([5,6,7,8],['arm_l3','arm_l2','arm_l1','arm_l0'])):
    axes[1].plot(jp[:, idx], label=name, linewidth=2)
axes[1].axhline(0.13, color='red', linestyle='--', alpha=0.7, label='max extension (0.13m)')
axes[1].set_title('Arm Extension Trajectory'); axes[1].set_ylabel('Extension (m)')
axes[1].legend(); axes[1].grid(alpha=0.3)
plt.suptitle('Stretch Pick — Phase Analysis (extend → lower → grasp → lift)', fontsize=13)
plt.tight_layout()
plt.savefig(DATA/'phase_analysis.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Cube Z trajectory — shows successful picks
fig, ax = plt.subplots(figsize=(12, 5))
for path in random.sample(episodes, min(10, len(episodes))):
    ep = np.load(path)
    ax.plot(ep['observation_cube_pos'][:, 2], alpha=0.6, linewidth=1.5)
ax.axhline(0.4, color='red', linestyle='--', linewidth=2, label='Success threshold (0.4m)')
ax.axhline(0.03, color='gray', linestyle=':', linewidth=1.5, label='Resting height (0.03m)')
ax.set_title('Cube Z Height During Pick — 10 Random Episodes')
ax.set_xlabel('Timestep'); ax.set_ylabel('Z Height (m)'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(DATA/'cube_z_trajectory.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Dataset statistics
lengths = [len(np.load(e)['reward']) for e in episodes]
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(lengths, bins=20, color='steelblue', alpha=0.7, edgecolor='black')
ax.set_title('Episode Length Distribution — Stretch Pick Cube')
ax.set_xlabel('Steps'); ax.set_ylabel('Count'); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(DATA/'dataset_stats.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Episodes: {len(episodes)}')
print(f'Mean length: {np.mean(lengths):.1f} | Total steps: {sum(lengths):,}')

In [ ]:
# Full dataset summary
datasets = {
    'Franka\nPick Cube':   '/home/shareef/robot_zoo/data/demos/franka_pick_cube',
    'Franka\nOpen Drawer': '/home/shareef/robot_zoo/data/demos/franka_open_drawer',
    'Franka\nReach':       '/home/shareef/robot_zoo/data/demos/franka_reach',
    'Stretch\nPick Cube':  '/home/shareef/robot_zoo/data/demos/stretch_pick_cube',
}
counts, steps_list = [], []
for path in datasets.values():
    eps = sorted(Path(path).glob('episode_*.npz'))
    counts.append(len(eps))
    steps_list.append(sum(len(np.load(e)['reward']) for e in eps))
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colors = ['#4C72B0','#4C72B0','#4C72B0','#DD8452']
axes[0].bar(datasets.keys(), counts, color=colors, edgecolor='black', alpha=0.85)
axes[0].set_title('Episodes per Task'); axes[0].set_ylabel('Count')
for i, v in enumerate(counts): axes[0].text(i, v+1, str(v), ha='center', fontweight='bold')
axes[1].bar(datasets.keys(), [s//1000 for s in steps_list], color=colors, edgecolor='black', alpha=0.85)
axes[1].set_title('Total Timesteps (thousands)'); axes[1].set_ylabel('K steps')
plt.suptitle('Robot Zoo — Week 4 Complete\nBlue=Franka  Orange=Stretch', fontsize=14)
plt.tight_layout()
plt.savefig(DATA/'week4_summary.png', dpi=120, bbox_inches='tight')
plt.show()
print('='*50)
print('WEEK 4 COMPLETE — Robot Zoo Dataset Summary')
print('='*50)
for name, c, s in zip(datasets.keys(), counts, steps_list):
    print(f'  {name.replace(chr(10)," "):22s}: {c:3d} demos | {s:6,} steps')
print(f'  {"TOTAL":22s}: {sum(counts):3d} demos | {sum(steps_list):6,} steps')